In [1]:
!pip install pandas scikit-learn matplotlib seaborn --break-system-packages

^C


In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('churn.db')

file_map = {
    'orders': 'orders.csv',
    'customers': 'customers.csv',
    'order_items': 'order_items.csv',
    'payments': 'payments.csv',
    'products': 'products.csv',
}

for table_name, filename in file_map.items():
    df = pd.read_csv(filename)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Loaded {table_name}: {len(df)} rows")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Loaded orders: 99441 rows
Loaded customers: 99441 rows
Loaded order_items: 112650 rows


KeyboardInterrupt: 

In [3]:
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE customer_orders AS
SELECT 
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    p.payment_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered';
""")

conn.commit()

OperationalError: table customer_orders already exists

Business question it answers: "Who are my customers, how recently did they buy, how often, and how much do they spend?"

In [ ]:
cursor.execute("""
CREATE Table rfm_base AS
SELECT 
    customer_unique_id,
    JULIANDAY((SELECT MAX(order_purchase_timestamp) FROM orders)) - JULIANDAY(MAX(order_purchase_timestamp)) AS recency_days,
    COUNT(DISTINCT order_id) AS frequency,
    SUM(payment_value) AS monetary
FROM customer_orders
GROUP BY customer_unique_id;
""")

conn.commit()

In [ ]:
import os

os.makedirs('outputs', exist_ok=True)

# 1. RFM base table
rfm_base = pd.read_sql("SELECT * FROM rfm_base", conn)
rfm_base.to_csv('outputs/rfm_base.csv', index=False)
print(f"rfm_base.csv saved: {len(rfm_base)} rows")

# 2. Cohort analysis
cohort_query = """
WITH first_purchase AS (
    SELECT customer_unique_id, MIN(DATE(order_purchase_timestamp, 'start of month')) AS cohort_month
    FROM customer_orders
    GROUP BY customer_unique_id
),
orders_with_cohort AS (
    SELECT 
        co.customer_unique_id,
        fp.cohort_month,
        DATE(co.order_purchase_timestamp, 'start of month') AS order_month
    FROM customer_orders co
    JOIN first_purchase fp ON co.customer_unique_id = fp.customer_unique_id
)
SELECT 
    cohort_month,
    order_month,
    COUNT(DISTINCT customer_unique_id) AS active_customers
FROM orders_with_cohort
GROUP BY cohort_month, order_month
ORDER BY cohort_month, order_month
"""
cohort_data = pd.read_sql(cohort_query, conn)
cohort_data.to_csv('outputs/cohort_analysis.csv', index=False)
print(f"cohort_analysis.csv saved: {len(cohort_data)} rows")

# 3. Category repeat-purchase
category_query = """
SELECT 
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    ROUND(COUNT(DISTINCT oi.order_id) * 1.0 / COUNT(DISTINCT o.customer_id), 2) AS orders_per_customer
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY orders_per_customer DESC
"""
category_data = pd.read_sql(category_query, conn)
category_data.to_csv('outputs/category_repeat_purchase.csv', index=False)
print(f"category_repeat_purchase.csv saved: {len(category_data)} rows")

print("\nAll files saved:", os.listdir('outputs'))

In [17]:
rfm = pd.read_csv("outputs/rfm_base.csv")

# Score each dimension into quintiles (1-5)
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5,4,3,2,1])   # lower recency_days = more recent = higher score
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5])

def segment(row):
    r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r <= 2 and m >= 4:
        return 'At Risk (High Value)'
    elif r <= 2 and f >= 3 and m <= 3:
        return 'At Risk (Low Value)'
    elif r >= 4 and f <= 2 and m >= 4:
        return 'Big Spenders (New)'
    elif r >= 4 and f <= 2 and m <= 3:
        return 'New/Promising'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost'
    else:
        return 'Potential Loyalist'
    
rfm['segment'] = rfm.apply(segment, axis=1)

In [20]:
summary = rfm.groupby('segment').agg(
    customer_count=('customer_unique_id', 'count'),
    total_revenue=('monetary', 'sum'),
    avg_revenue_per_customer=('monetary', 'mean')
).sort_values('total_revenue', ascending=False)

print(summary)

                      customer_count  total_revenue  avg_revenue_per_customer
segment                                                                      
At Risk (High Value)           14516     2734856.93                188.402930
Loyal Customers                14064     2026008.90                144.056378
Potential Loyalist             23635     1590732.21                 67.304092
Champions                       6419     1270355.92                197.905580
Big Spenders (New)              5970     1089972.21                182.574910
At Risk (Low Value)            13501      777607.46                 57.596286
New/Promising                   8991      512115.38                 56.958668
Lost                            6261      250433.20                 39.998914


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Define churn: no purchase in 90+ days (a business-defined threshold, not a technical one — 
# state clearly in your writeup that this threshold is a judgment call and could be tuned per business)
rfm['churned'] = (rfm['recency_days'] > 90).astype(int)

X = rfm[['frequency', 'monetary']]  # recency deliberately excluded — see note below
y = rfm['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, preds))
print("AUC:", roc_auc_score(y_test, probs))
print("Feature importance:", dict(zip(X.columns, model.feature_importances_)))

NameError: name 'rfm' is not defined

In [19]:
rfm['churn_probability'] = model.predict_proba(rfm[['frequency','monetary']])[:,1]
rfm.to_csv('customer_scored.csv', index=False)

In [1]:
from sklearn.metrics import roc_auc_score

# Predicted probabilities for the positive class
y_prob = model.predict_proba(X_test)[:, 1]

# ROC-AUC
auc = roc_auc_score(y_test, y_prob)

print(f"ROC-AUC: {auc:.3f}")

NameError: name 'model' is not defined